# Recon 4
GitHub token helper, neighbor pods, blender internals.

In [ ]:
import subprocess
cmds = [
    "echo '== curl localhost:4274 =='; curl -s --max-time 5 http://localhost:4274/ 2>&1 | head -c 4000; echo; curl -s --max-time 5 http://127.0.0.1:4274/ 2>&1 | head -c 4000",
    "echo '== headers =='; curl -s -i --max-time 5 http://127.0.0.1:4274/ 2>&1 | head -20",
    "for p in 4274 4275 4276; do echo == $p ==; curl -s -i --max-time 4 http://127.0.0.1:$p/ 2>&1 | head -8; done",
]
for c in cmds:
    print(f"$ {c}")
    p = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=40)
    print(p.stdout[:4000], p.stderr[:2000])
    print("---")

In [ ]:
import subprocess
cmds = [
    "echo '== entry_point_finder =='; head -100 /tmp/vivid-blender/entry_point_finder.py 2>&1",
    "echo '== collect_py_info =='; cat /tmp/vivid-blender/collect_py_info.py 2>&1",
    "echo '== requirements.txt =='; cat /tmp/vivid-blender/requirements.txt 2>&1",
    "echo '== /proc/1/environ =='; tr '\0' '\n' < /proc/1/environ 2>&1 | head -40",
    "echo '== search token files =='; find / -name '*token*' -o -name '*oidc*' -o -name '*credential*' 2>/dev/null | grep -v '/cloud/lib/venv' | grep -v '/proc/' | head -30",
    "echo '== find oidc =='; ls -la /var/run/secrets 2>&1; find / -path /proc -prune -o -name 'token' -print 2>/dev/null | grep -v venv | grep -v '/proc/' | head",
]
for c in cmds:
    print(f"$ {c}")
    p = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=40)
    print(p.stdout[:7000], p.stderr[:2000])
    print("---")

In [ ]:
import subprocess
script = r'''
import socket, urllib.request, concurrent.futures
def probe(ip):
    out = []
    for port in [3838, 8012, 4274, 8080, 8000, 9090, 8888, 5000, 22]:
        try:
            s = socket.create_connection((ip, port), timeout=0.5)
            out.append(port)
            s.close()
        except Exception:
            pass
    if out:
        return (ip, out)
    return None
targets = [f"192.168.4.{i}" for i in range(1,255) if i != 124]
with concurrent.futures.ThreadPoolExecutor(80) as ex:
    for r in ex.map(probe, targets):
        if r: print("OPEN", r)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, text=True, timeout=150)
print(p.stdout[:8000], p.stderr[:3000])

In [ ]:
import subprocess
script = r'''
import urllib.request
def req(url):
    try:
        resp = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"}), timeout=5)
        print(url, "->", resp.status, resp.headers.get("content-type"), resp.read(600)[:400])
    except Exception as e:
        print(url, "FAIL:", e)
for ip in ["192.168.4.13"]:
    req(f"http://{ip}:8012/")
    req(f"http://{ip}:3838/")
    req(f"http://{ip}:4274/")
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, text=True, timeout=90)
print(p.stdout[:8000], p.stderr[:3000])